# Compare forest cover models

Compare logistic regression, a decision tree, and a random forest. Use a stratified 80/20 split, train-only cross-validation, and macro-F1 so smaller cover classes count equally in model selection.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import make_column_transformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 0
data = pd.read_csv(Path("data") / "forest_cover_prepared.csv")
y = data.pop("Forest_Cover")
X = data
categories = ["Area", "Soil_Type"]
terrain = [column for column in X if column not in categories]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED,
)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)


def model_pipeline(estimator):
    preprocess = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), terrain),
        (OneHotEncoder(handle_unknown="ignore"), categories),
    )
    return make_pipeline(preprocess, estimator)


In [ ]:
models = {
    "Logistic regression": model_pipeline(LogisticRegression(max_iter=1_000)),
    "Decision tree": RandomizedSearchCV(
        model_pipeline(DecisionTreeClassifier(random_state=SEED)),
        {
            "decisiontreeclassifier__max_depth": [8, 14, 20],
            "decisiontreeclassifier__min_samples_leaf": [1, 3, 8],
        },
        n_iter=5, scoring="f1_macro", cv=cv, random_state=SEED,
    ),
    "Random forest": RandomizedSearchCV(
        model_pipeline(RandomForestClassifier(n_estimators=80, random_state=SEED)),
        {
            "randomforestclassifier__max_depth": [12, 20, None],
            "randomforestclassifier__min_samples_leaf": [1, 3, 8],
            "randomforestclassifier__max_features": ["sqrt", 0.5],
        },
        n_iter=5, scoring="f1_macro", cv=cv, random_state=SEED,
    ),
}
results = []
predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_f1": f1_score(y_test, pred, average="macro"),
    })

results = pd.DataFrame(results).set_index("model")
display(results.round(4))


In [ ]:
forest_predictions = predictions["Random forest"]
print(classification_report(y_test, forest_predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(
    y_test, forest_predictions, normalize="true", values_format=".2f", cmap="Blues",
)
plt.title("Random forest recall by true cover class")
plt.tight_layout()
plt.show()
